# ✈️ Naive PDF Reader by Collins Aerospace
### Text-to-Text Naive RAG using LangGraph + Groq


In [1]:
# Install necessary libraries
!pip install -q langchain langgraph langchain-community langchain-core pypdf chromadb sentence-transformers groq gradio

In [3]:
# Import libraries
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph
from typing import TypedDict
from groq import Groq
import gradio as gr

In [4]:
# Set Groq API Key
os.environ['GROQ_API_KEY'] = 'gSk_akED1vwdMChrWKLC1 MOmWGdyb3FY5uRW16LCg70xaLUhTVQZAa4h'
client = Groq()

In [5]:
# LLM Call Function
def call_llm(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama3-8b-8192"
    )
    return response.choices[0].message.content

In [6]:
# Graph State
class GraphState(TypedDict):
    question: str
    context: str
    answer: str

In [7]:
# Load & Index PDF
def load_and_index(pdf_path):
    loader = PyPDFLoader(pdf_path.name)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings()
    vectorstore = Chroma.from_documents(chunks, embeddings)

    return vectorstore.as_retriever()

In [8]:
# Retrieve Step
def retrieve(state):
    docs = retriever.get_relevant_documents(state['question'])
    context = "\n".join([doc.page_content for doc in docs])
    return {"context": context}

In [9]:
# Generate Step
def generate(state):
    prompt = f"""
    You are an aerospace AI expert.

    Context:
    {state['context']}

    Question:
    {state['question']}

    Provide structured output:
    - Summary
    - Risk Factors
    - Prevention Strategy
    - Final Recommendation
    """

    answer = call_llm(prompt)
    return {"answer": answer}

In [10]:
# Build LangGraph
graph = StateGraph(GraphState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
app = graph.compile()

In [11]:
# Evaluation Metrics
def evaluate_rag(question, answer, context):
    relevance = len(answer) / (len(context) + 1)
    faithfulness = "High" if answer in context else "Moderate"
    return {"Relevance": relevance, "Faithfulness": faithfulness}

In [12]:
# Pipeline Function
def rag_pipeline(pdf, question):
    global retriever
    # retriever = load_and_index(pdf.name)
    retriever = load_and_index(pdf)
    result = app.invoke({"question": question})

    metrics = evaluate_rag(question, result['answer'], result.get('context', ''))

    return f"""
    ✨ Answer:
    {result['answer']}

    📊 Metrics:
    Relevance: {metrics['Relevance']}
    Faithfulness: {metrics['Faithfulness']}
    """

In [15]:
# Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# ✈️ Naive PDF Reader by Collins Aerospace")

    gr.HTML("""
    <div style='overflow:hidden;'>
      <div style='animation: marquee 50s linear infinite;'>🚀 PDF Reader by Collins Aerospace</div>
    </div>
    <style>
    @keyframes marquee {0%{transform:translateX(100%);}100%{transform:translateX(-100%);}}
    </style>
    """)

    pdf_input = gr.File(label="Upload PDF")
    question = gr.Textbox(label="Ask Question")
    output = gr.Textbox(lines=15)

    btn = gr.Button("Generate")
    btn.click(rag_pipeline, inputs=[pdf_input, question], outputs=output)

demo.launch(share=True, debug=True)

/tmp/ipykernel_1820/4252166928.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://45be2577bf7ea6e6d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AttributeError: module 'gradio' has no attribute 'blocks'